# Ernesto Investing AI — Notebook 02: Clasificador SVC

Entrena un Support Vector Classifier con GridSearchCV para predecir
senales BUY/SELL sobre los 5 tickers del proyecto. Lee de la coleccion
`precios_ohlcv` (poblada por el Notebook 01) y guarda resultados en
`predicciones` y `metricas_modelos`.

**Requisito previo:** el Notebook 01 debe haberse ejecutado al menos
una vez para que existan datos en `precios_ohlcv`.

**Referencia:** inspirado en la investigacion doctoral del profesor
sobre SVM con optimizacion multi-empresa por GridSearch.

## 1. Instalacion e importaciones

In [ ]:
!pip install "pymongo[srv]" --quiet

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timezone

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from pymongo import MongoClient
from google.colab import userdata

## 2. Configuracion y conexion a MongoDB

In [ ]:
TICKERS = ["FSM", "VOLCABC1.LM", "ABX.TO", "BVN", "BHP"]

MONGO_URI = userdata.get("MONGO_URI")
MONGO_DB_NAME = "ernesto_investing_ai"

cliente = MongoClient(MONGO_URI)
db = cliente[MONGO_DB_NAME]

col_precios = db["precios_ohlcv"]
col_predicciones = db["predicciones"]
col_metricas = db["metricas_modelos"]

print("Conexion exitosa. Documentos en precios_ohlcv:", col_precios.count_documents({}))

## 3. Carga de datos desde MongoDB

Lee los precios ya calculados por el Notebook 01 (no se vuelve a
descargar de yfinance ni se recalculan indicadores: eso ya vive en
`precios_ohlcv`).

In [ ]:
def cargar_precios(ticker: str) -> pd.DataFrame:
    """Lee todos los documentos de un ticker desde precios_ohlcv, ordenados por fecha."""
    cursor = col_precios.find({"ticker": ticker}).sort("fecha", 1)
    filas = []
    for doc in cursor:
        fila = {
            "fecha": doc["fecha"],
            "open": doc["open"], "high": doc["high"], "low": doc["low"],
            "close": doc["close"], "volume": doc["volume"],
        }
        fila.update(doc["indicadores"])
        filas.append(fila)
    return pd.DataFrame(filas)

## 4. Ingenieria de caracteristicas y variable objetivo

Variable objetivo binaria (`Trend`): 1 = BUY si el precio de cierre
del dia siguiente es mayor que el de hoy, 0 = SELL en caso contrario.
Se descartan filas con indicadores en NaN (dias iniciales sin
suficiente historia) y la ultima fila (no tiene "dia siguiente" para
calcular el target real, se guarda aparte para la prediccion actual).

In [ ]:
def construir_features_y_target(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    """
    Devuelve (X, fila_actual, y):
      X          -> features listas para entrenar (sin la ultima fila)
      fila_actual -> ultima fila disponible, para generar la senal de HOY
      y          -> target binario alineado con X
    """
    df = df.copy()
    df["retorno_diario"] = df["close"].pct_change()
    df["distancia_sma20"] = (df["close"] - df["sma_20"]) / df["sma_20"]
    df["distancia_sma50"] = (df["close"] - df["sma_50"]) / df["sma_50"]
    df["ancho_bollinger"] = (df["bb_upper"] - df["bb_lower"]) / df["close"]
    df["volatilidad_5d"] = df["retorno_diario"].rolling(5).std()

    df["target"] = (df["close"].shift(-1) > df["close"]).astype(int)

    columnas_features = [
        "sma_20", "sma_50", "ema_12", "ema_26", "rsi_14",
        "macd", "macd_signal", "macd_hist", "bb_upper", "bb_lower",
        "retorno_diario", "distancia_sma20", "distancia_sma50",
        "ancho_bollinger", "volatilidad_5d",
    ]

    df_limpio = df.dropna(subset=columnas_features)

    fila_actual = df_limpio.iloc[[-1]][columnas_features]
    df_entrenamiento = df_limpio.iloc[:-1]

    X = df_entrenamiento[columnas_features]
    y = df_entrenamiento["target"]
    return X, fila_actual, y

## 5. Entrenamiento con GridSearchCV

Particion temporal 80/20 (sin mezclar filas, para no filtrar
informacion del futuro hacia el pasado). El escalador se ajusta
**solo** con datos de entrenamiento. `TimeSeriesSplit` en la
validacion cruzada evita usar datos futuros para elegir
hiperparametros.

In [ ]:
def entrenar_svc(X: pd.DataFrame, y: pd.Series):
    """Entrena un SVC con GridSearchCV y devuelve el mejor modelo, el scaler y metricas de test."""
    n_train = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:n_train], X.iloc[n_train:]
    y_train, y_test = y.iloc[:n_train], y.iloc[n_train:]

    scaler = StandardScaler()
    X_train_esc = scaler.fit_transform(X_train)
    X_test_esc = scaler.transform(X_test)

    grilla = {
        "kernel": ["linear", "rbf", "poly", "sigmoid"],
        "C": [0.1, 1, 10, 100],
        "gamma": ["scale", "auto"],
    }
    busqueda = GridSearchCV(
        SVC(probability=True),
        grilla,
        cv=TimeSeriesSplit(n_splits=3),
        scoring="f1",
        n_jobs=-1,
    )
    busqueda.fit(X_train_esc, y_train)

    mejor_modelo = busqueda.best_estimator_
    predicciones_test = mejor_modelo.predict(X_test_esc)

    metricas = {
        "accuracy": float(accuracy_score(y_test, predicciones_test)),
        "precision": float(precision_score(y_test, predicciones_test, zero_division=0)),
        "recall": float(recall_score(y_test, predicciones_test, zero_division=0)),
        "f1": float(f1_score(y_test, predicciones_test, zero_division=0)),
        "rmse": None, "mae": None, "r2": None,
    }
    matriz = confusion_matrix(y_test, predicciones_test).tolist()

    return mejor_modelo, scaler, busqueda.best_params_, metricas, matriz

## 6. Ejecucion completa: entrenar, predecir y guardar en MongoDB

In [ ]:
resumen = {}

for ticker in TICKERS:
    print(f"Procesando {ticker}...")
    try:
        df = cargar_precios(ticker)
        if len(df) < 60:
            print(f"  {ticker}: insuficientes datos ({len(df)} filas), se omite")
            continue

        X, fila_actual, y = construir_features_y_target(df)
        modelo, scaler, mejores_parametros, metricas, matriz = entrenar_svc(X, y)

        # Prediccion de la senal actual (mas reciente disponible)
        fila_actual_esc = scaler.transform(fila_actual)
        senal_predicha = int(modelo.predict(fila_actual_esc)[0])
        probabilidad = float(modelo.predict_proba(fila_actual_esc)[0][senal_predicha])

        ahora = datetime.now(timezone.utc)

        col_predicciones.insert_one({
            "ticker": ticker,
            "modelo": "svc",
            "tipo": "clasificacion",
            "fecha_generacion": ahora,
            "senal": "BUY" if senal_predicha == 1 else "SELL",
            "probabilidad": round(probabilidad, 4),
            "precio_predicho": None,
            "horizonte_dias": None,
            "banda_confianza": {"inferior": None, "superior": None},
        })

        col_metricas.update_one(
            {"ticker": ticker, "modelo": "svc"},
            {"$set": {
                "ticker": ticker,
                "modelo": "svc",
                "fecha_entrenamiento": ahora,
                "metricas": metricas,
                "hiperparametros": mejores_parametros,
                "matriz_confusion": matriz,
            }},
            upsert=True,
        )

        resumen[ticker] = {"senal": "BUY" if senal_predicha == 1 else "SELL", "accuracy": round(metricas["accuracy"], 3)}
        print(f"  {ticker}: senal={resumen[ticker]['senal']}, accuracy={resumen[ticker]['accuracy']}, "
              f"mejores_parametros={mejores_parametros}")

    except Exception as error:
        print(f"  ERROR con {ticker}: {error}")
        resumen[ticker] = f"ERROR: {error}"

print()
print("Resumen final:", resumen)

## 7. Verificacion final

In [ ]:
print("Total de predicciones SVC guardadas:", col_predicciones.count_documents({"modelo": "svc"}))
print("Total de metricas SVC guardadas:", col_metricas.count_documents({"modelo": "svc"}))
print()

for ticker in TICKERS:
    ultima = col_predicciones.find_one({"ticker": ticker, "modelo": "svc"}, sort=[("fecha_generacion", -1)])
    if ultima:
        print(f"{ticker}: senal={ultima['senal']}, probabilidad={ultima['probabilidad']}")